# Running a tabular foundation model, live

The companion notebook, [`tutorial.ipynb`](tutorial.ipynb), reads the foundation-model
results from the committed sweep. This one runs them **live**: TabDPT and TabICL fit and
predict here, in the notebook, and the paired-bootstrap tie is computed on the spot.

TabDPT and TabICL run on a CPU in seconds. **TabFM** tells the same story but is a
six-hour CPU job, so it stays in `scripts/sweep.py`.

> **Two setup notes.** The foundation-model libraries pull in torch and FAISS, and on
> macOS their OpenMP runtimes can deadlock or crash a Jupyter kernel when left
> multi-threaded, so the first cell pins everything to one thread. For the same reason the
> tree in the tie below is read from the committed sweep rather than run here: LightGBM
> brings its own OpenMP runtime, and running it alongside torch in one kernel is what
> crashes it. The first run also downloads the model weights; later runs use the cache.

In [1]:
import os
# Keep the kernel stable: one OpenMP thread across torch and FAISS.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

here = Path.cwd()
repo = next(p for p in [here, *here.parents] if (p / "scripts" / "prep.py").exists())
sys.path.insert(0, str(repo / "scripts"))

import torch; torch.set_num_threads(1)
from prep import load_split, encode_for_gbdt
import run_one as ro
from showdown import paired

warnings.filterwarnings("ignore")
print("repo:", repo.name, "| torch threads:", torch.get_num_threads())

repo: tabular-fm-scorecard | torch threads: 1


## Run the foundation models, live

FICO home equity, capped at 1,000 rows, which is what an in-context model can afford.
`fit` here does not train weights: it reads the rows into context, and the work happens at
predict time, fresh on every call. That is why the cost lands where it does. TabDPT takes
a numeric matrix; TabICL takes the raw frame.

In [2]:
def score_1m(seconds_per_1k):
    s = seconds_per_1k * 1000
    if s < 90:     return f"{s:.1f} s"
    if s < 5400:   return f"{s/60:.1f} min"
    if s < 172800: return f"{s/3600:.1f} h"
    return f"{s/86400:.1f} days"

Xtr, ytr, Xte, yte = load_split("heloc", n_context=1000, n_test=1000)
Xnum_tr, Xnum_te = encode_for_gbdt(Xtr, Xte)     # numeric frame for TabDPT
print(f"train {len(ytr)} rows, test {len(yte)} rows, positive rate {yte.mean():.1%}\n")

rows = []
fm_proba = None
for key, (a, b) in {"tabdpt": (Xnum_tr, Xnum_te), "tabicl": (Xtr, Xte)}.items():
    proba, t_fit, t_pred, _ = ro.RUNNERS[key](a, ytr, b)
    if key == "tabdpt":
        fm_proba = proba
    rows.append({"model": key, "AUC": round(roc_auc_score(yte, proba), 4),
                 "fit s": round(t_fit, 2), "predict s / 1k": round(t_pred / len(yte) * 1000, 3),
                 "score 1M rows": score_1m(t_pred / len(yte) * 1000)})
pd.DataFrame(rows)

train 1000 rows, test 1000 rows, positive rate 47.8%



,model,AUC,fit s,predict s / 1k,score 1M rows
0,tabdpt,0.8104,0.00,0.739,12.3 min
1,tabicl,0.8217,0.12,7.040,2.0 h


Two models sold under the same label, and they neither agree with each other nor cost
the same. That is the first thing to know about the category. Even TabDPT, the cheap one,
is already far slower per prediction than a tree, and TabFM is three to four orders of
magnitude beyond that.

## The catch: give the tree its full data

Now the comparison a business actually faces: the same held-out rows, against a tree
trained on **every** row the dataset has, not the 1,000 the foundation model was capped
at. We read that tree's predictions from the committed sweep so LightGBM's OpenMP runtime
stays out of this kernel, and confirm it was scored on the identical test rows.

In [3]:
tree = np.load(repo / "results" / "probe" / "heloc__lightgbm_full__c9459.npz")
assert np.array_equal(tree["y_true"], yte), "committed tree used different test rows"
tree_proba = tree["proba"]

pd.DataFrame([
    {"model": "TabDPT (foundation)", "rows": len(ytr), "AUC": round(roc_auc_score(yte, fm_proba), 4)},
    {"model": "LightGBM (tree)",     "rows": 9459,      "AUC": round(roc_auc_score(yte, tree_proba), 4)},
])

,model,rows,AUC
0,TabDPT (foundation),1000,0.8104
1,LightGBM (tree),9459,0.8149


The tree, given its rows, sits right next to the live foundation model. Whether the
gap is real is a question a single AUC cannot answer, so we run the paired bootstrap on the
identical test rows and read the interval like an election poll.

In [4]:
r = paired(yte, fm_proba, tree_proba)      # gap = TabDPT minus tree, with 95% CI
print(f"TabDPT (1,000 rows) minus LightGBM (all 9,459 rows): {r['delta']:+.4f} AUC")
print(f"95% interval on the gap: [{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]   p = {r['p']}")
print("verdict:", "TIE  (the interval crosses zero)" if r['ci_lo'] <= 0 <= r['ci_hi'] else "separated")

TabDPT (1,000 rows) minus LightGBM (all 9,459 rows): -0.0046 AUC
95% interval on the gap: [-0.0190, +0.0103]   p = 0.512
verdict: TIE  (the interval crosses zero)


**Computed live, and it is a tie.** Whatever edge the foundation model had at 1,000
rows sits inside the margin of error once the tree has the data it would actually have in
production. That is the whole study in one cell, and TabFM would only make it more
expensive: it draws the same tie at far higher cost per prediction.

- **Quality:** a draw, once the tree is given its data.
- **Time:** the foundation model wins. Nothing to tune, retrain, or maintain.
- **Cost:** the foundation model loses, and the gap only widens with TabFM.

Full method and every number: [FINDINGS.md](../FINDINGS.md). To run the whole sweep,
including TabFM and the trees:

```bash
python scripts/sweep.py --models logreg,xgboost,lightgbm,catboost,tabicl,tabdpt
python scripts/compare.py && python scripts/verify_readme_claims.py
```